# Hyperparameter Tuning

**Objective:** Learn hyperparameter tuning with simple, step-by-step examples.

## Parameters vs hyperparameters

A model parameter is learned from data, such as a linear-regression coefficient. A hyperparameter is configured by us, such as tree depth, number of neighbors or regularization strength. Hyperparameter tuning searches for settings that work well on validation data.

## Hyperparameter Tuning
Hyperparameters are settings chosen before training. GridSearchCV and RandomizedSearchCV help search for good settings.

### Grid search and random search

GridSearchCV evaluates every combination in a supplied parameter grid using cross-validation. RandomizedSearchCV samples a fixed number of combinations from distributions or lists. Grid search is thorough for small grids; random search explores larger spaces more economically.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

data = pd.read_csv('datasets/iris.csv')
X = data.drop(columns=['species'])
y = data['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

### Bayesian optimization intuition

Bayesian optimization builds a probabilistic approximation of how hyperparameter choices relate to validation scores. It uses previous trials to choose promising next trials. It can reduce expensive experiments but requires additional tooling and careful search-space design.

In [2]:
parameter_grid = {
    'n_estimators': [50, 100],
    'max_depth': [2, 4, None]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    parameter_grid,
    cv=3,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)

{'max_depth': 2, 'n_estimators': 50}


## Avoid tuning on the test set

The validation score guides model choice. Repeatedly trying settings based on the final test score leaks information from the test set into development. Use cross-validation or a dedicated validation set for tuning, then evaluate once on the untouched test set.

In [3]:
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, predictions))

Test accuracy: 1.0


## Key terms

- **Search space:** allowed values or ranges for hyperparameters.
- **Grid search:** evaluate all specified combinations.
- **Random search:** evaluate sampled combinations.
- **Bayesian optimization:** use past trials to guide future trials.
- **Nested cross-validation:** an outer loop estimates generalization while an inner loop tunes hyperparameters.

In [4]:
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=parameter_grid,
    n_iter=3,
    cv=3,
    random_state=RANDOM_STATE
)
random_search.fit(X_train, y_train)
print(random_search.best_params_)

{'n_estimators': 50, 'max_depth': 2}


## Try This Yourself
- Add `min_samples_split` to the grid.
- Increase `n_iter` for RandomizedSearchCV.
- Compare grid search and random search results.